1. **Primary tutorial to follow: Hugging Face fine-tuning**
   - Use the Hugging Face workflow as the implementation path for sentence classification: load labeled data → tokenize → split data → load a pretrained sequence-classification model → fine-tune → evaluate → save → infer.
   - This directly matches the downstream sentence-classification task. BERT is designed to be fine-tuned by adding a task-specific output layer and updating the pretrained encoder parameters together with that layer [1].

2. **Keep from the D2L BERT tutorial: conceptual foundations**
   - Keep the BERT input format: `[CLS] sentence [SEP]` for a single sentence.
   - Keep the meaning of token, segment, and positional embeddings.
   - Keep the role of the `[CLS]` representation: it is the sequence-level representation used by the classification head.
   - Keep the distinction between pretraining and fine-tuning:
     - Pretraining: MLM + NSP on unlabeled text.
     - Fine-tuning: supervised classification using your sentence labels.
   - Keep the idea that BERT representations are context-sensitive: the same token may receive a different vector in a different sentence context [1].

3. **Do not carry forward from D2L for ordinary sentence classification**
   - Do not train BERT from scratch with `BERTEncoder`, `MaskLM`, `NextSentencePred`, `_get_batch_loss_bert`, or `train_bert`.
   - Do not use NSP labels, MLM labels, MLM masking, or the WikiText pretraining dataset for your supervised sentence-classification experiment.
   - Those components belong to BERT pretraining, not downstream sentence classification [1].

4. **Define the classification problem before preprocessing**
   - Identify the text column: currently `Base Sentence`.
   - Identify the ground-truth label column.
   - Confirm whether the task is binary classification (`0` / `1`) or multiclass classification.
   - Define a label mapping, such as `0 = negative`, `1 = positive`, and retain it for training, reporting, and inference.

5. **Keep data-quality checks from your current notebook**
   - Check for missing sentences.
   - Check that every retained sentence is a string.
   - Check for missing or invalid labels.
   - Check class counts before splitting, so you know whether the classes are balanced.

6. **Create proper train, validation, and test splits**
   - Training split: used to update model parameters.
   - Validation split: used during development for monitoring metrics and selecting settings.
   - Test split: held out until the final evaluation.
   - Your current `test` file is functioning as evaluation data; reserve it as a final test set if possible, and create validation data from the training set.

7. **Use one matched tokenizer–model family**
   - The tokenizer and model must come from the same checkpoint family.
   - Your notebook currently initializes a DistilBERT tokenizer but loads `bert-base-cased` as the model; these should be aligned.
   - Decide on one pretrained checkpoint and use its corresponding tokenizer and sequence-classification model.

8. **Tokenize the sentence column**
   - Tokenization converts each sentence into model inputs such as token IDs and attention masks.
   - Apply truncation to enforce a maximum sequence length.
   - Use padding so examples in a batch have compatible lengths.
   - Segment IDs are generally unnecessary for your single-sentence task; the Hugging Face tokenizer/model handles the expected inputs.

9. **Fine-tune a sequence-classification model**
   - Use `AutoModelForSequenceClassification` with `num_labels` equal to the number of classes.
   - This model consists of a pretrained BERT encoder plus a classification head that maps the `[CLS]`-derived sequence representation to class logits.
   - Fine-tuning updates the pretrained BERT parameters and trains the classification head for your labels [1].

10. **Configure training deliberately**
    - Specify output/checkpoint location.
    - Specify epochs, learning rate, batch sizes, evaluation frequency, logging, and checkpoint-saving strategy.
    - Start with a small training subset only as a pipeline check.
    - Run the final experiment on the full training split after the pipeline works.

11. **Evaluate beyond accuracy**
    - Keep your `EvaluationMetric` utilities for the final evaluation.
    - Report accuracy, precision, recall, F1 score, classification report, and confusion matrix.
    - For binary classification, report TN, FP, FN, and TP.
    - If classes are imbalanced, emphasize per-class precision, recall, F1, and macro/weighted F1 rather than accuracy alone.

12. **Perform final test evaluation once**
    - After choosing the model configuration using validation results, evaluate the selected model once on the held-out test set.
    - Store predictions, true labels, predicted probabilities or logits, and evaluation metrics for error analysis.

13. **Conduct error analysis**
    - Review false positives and false negatives.
    - Look for ambiguous sentences, label noise, very short inputs, long inputs that were truncated, and class-specific language patterns.
    - Use these findings to improve data, label definitions, or model settings.

14. **Save reproducible artifacts**
    - Save the fine-tuned model, matched tokenizer, label mapping, training configuration, evaluation report, confusion matrix, and test predictions.
    - Record the random seed, dataset version, model checkpoint, maximum length, and split definitions.

15. **Use D2L only if your goal changes**
    - Return to the D2L pretraining workflow only if you specifically want to learn or experiment with pretraining a BERT-like model from scratch on a large unlabeled corpus.
    - For your current labeled sentence-classification task, Hugging Face fine-tuning is the correct main workflow.